In [ ]:
%load_ext autoreload
%autoreload 2

import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys

# 1. Get the absolute path of the current script's directory
current_dir = os.getcwd()

# 2. Get the path of the parent directory
parent_dir = os.path.dirname(current_dir)

# 3. Add the parent directory to the top of sys.path
sys.path.insert(0, parent_dir)

import baselines,metrics,features,models


In [ ]:
print(metrics.__file__)
print([name for name in dir(metrics) if not name.startswith("_")])


In [ ]:
print(baselines.__file__)
print([name for name in dir(baselines) if not name.startswith("_")])


In [ ]:
data = yf.download("^GSPC", start="1990-01-01", auto_adjust=True)


log_returns = np.log(data["Close"]["^GSPC"]).diff().iloc[1:]


sq_returns = np.square(log_returns)
vol_of_5days = np.sqrt(sq_returns.rolling(5).mean()).shift(-5).iloc[:log_returns.shape[0] - 5]


train = vol_of_5days.loc["1990" : "2018"]
val = vol_of_5days.loc["2019" : "2022"]
test = vol_of_5days.loc["2023" : ]


print(train.head())
print(train.tail())
print(val.head())
print(val.tail())
print(test.head())
print(test.tail())


In [ ]:
print(data.columns)


In [ ]:
pred = pd.Series([1])
act = pd.Series([0.5])

print(metrics.QLIKE(pred,act))

In [ ]:
print(val.head())

In [ ]:
import inspect
print(baselines.__file__)
print(inspect.getsource(baselines.garch))

In [ ]:
print(type(data))
garch = baselines.garch(log_returns,"2019-01-02")

In [ ]:
naive = baselines.naive(log_returns)
rolling = baselines.rolling(log_returns)
ewm = baselines.ewm(log_returns)

print(metrics.QLIKE(garch,val)) 
print(metrics.QLIKE(naive,val))
print(metrics.QLIKE(rolling,val))
print(metrics.QLIKE(ewm,val))

In [ ]:
df = features.creater_features(data)

train = df.loc["1990" : "2018"]
val = df.loc["2019" : "2022"]
test = df.loc["2023" : ]

print(train.shape)

train = train[train["today_vol"] != 0]

print(train.shape)


In [ ]:
q = models.HAR(train,val)  
qlog = models.HAR_LOG(train,val)
print(metrics.QLIKE(q,val["target"]))
print(metrics.QLIKE(qlog,val["target"]))
print(metrics.RMSE(q,val["target"]))
print(metrics.RMSE(garch,val["target"]))
print(metrics.RMSE(qlog,val["target"]))

In [ ]:
print((qlog < val["target"]).mean())   # share of days under-predicted
print(qlog.mean(), val["target"].mean())

In [ ]:
cols = ["today_vol", "past_5_days", "past_22_days"]

for leaf in [5,10,20,40,50,65,100,200]:
    rf = models.random_forest(train,val,cols,leaf)
    print(metrics.QLIKE(rf,val["target"]))



for leaf in [5,10,20,40,50,65,100,200]:
    rf = models.random_forest(train,val,cols,leaf)
    print(metrics.RMSE(rf,val["target"]))




In [ ]:

    for lr in [0.01,0.05,0.1]:
        for depth in [1,2,3,4,5]:
            xgb = models.XGBOOST(train,val,cols,depth,lr)
            score = metrics.QLIKE(xgb,val["target"])

            print(f"for lr={lr}, depth = {depth}, the score is {score}")

In [ ]:
    for lr in [0.01,0.05,0.1]:
        for depth in [1,2,3,4,5]:
            xgb = models.XGBOOST_QLIKE(train,val,cols,depth,lr)
            score = metrics.QLIKE(xgb,val["target"])
            

            print(f"for lr={lr}, depth = {depth}, the score is {score}")

In [ ]:
xg1 = models.XGBOOST_QLIKE(train,val,cols,1,0.01)
xg2 = models.XGBOOST(train,val,cols,1,0.01)
print(metrics.RMSE(xg1,val["target"]))
print(metrics.RMSE(xg2,val["target"]))

In [ ]:
both = pd.concat([xg1,val],axis = 1,join='inner').dropna()
print(both.iloc[:,0].mean(), both.iloc[:,1].mean())

FEATURE SELECTION AND VALIDATION

In [ ]:
har = models.HAR(train,val)

cols = ["today_vol","past_5_days", "past_22_days","high_low_1","vix"]
rf_new = models.random_forest(train,val,cols,65)
xgb_new = models.XGBOOST(train,val,cols,lr=0.05,depth=1)
xgb_qlike = models.XGBOOST_QLIKE(train,val,cols,lr=0.01,depth=1)


for mod in [har,rf_new,xgb_new,xgb_qlike]:
      print(metrics.QLIKE(mod,val["target"]))

In [ ]:
print(df.head(10))


In [ ]:

for mod in [garch,xgb_new,xgb_qlike]:
 for i in ["2019","2020", "2021","2022"]:
     print(metrics.QLIKE(mod,val.loc[i]["target"]))

In [ ]:
window = slice("2020-02-01", "2020-04-30")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(val.loc[window, "target"], label="Actual (next-week vol)", color="black")
ax.plot(garch.loc[window], label="GARCH", linestyle="--")
ax.plot(xgb_qlike.loc[window], label="XGBoost (QLIKE loss)")
ax.set_ylabel("Volatility (daily)")
ax.set_title("Forecasts vs actual, Feb–Apr 2020")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
for c in ["today_vol", "high_low_1"]:
    print(f"{c}: train max = {train[c].max():.4f}, 2020 max = {val.loc['2020', c].max():.4f}")
print(f"Highest XGBoost forecast in 2020: {xgb_qlike.loc['2020'].max():.4f}")
print(f"Highest actual target in 2020:    {val.loc['2020', 'target'].max():.4f}")


print(garch)

In [ ]:
gv = baselines.garch_variance(log_returns, "2019-01-02")       # same returns and cutoff as your baseline
xgb_hybrid = models.XGBOOST_HYBRID(train, val, cols, depth=1, lr=0.01,
                                   garch_var_train=gv, garch_var_val=gv)
print(metrics.QLIKE(xgb_hybrid,val["target"]))

for i in ["2019","2020", "2021","2022"]:
     print(metrics.QLIKE(xgb_hybrid,val.loc[i]["target"]))


In [ ]:
o, c = data["Open"]["^GSPC"], data["Close"]["^GSPC"]
stale = (o == c.shift(1))
print(stale.groupby(stale.index.year).mean().round(2))

In [ ]:
for lr in [0.01,0.05,0.1]:
    for depth in [1,2,3,4,5]:
        gb = models.XGBOOST_QLIKE(train,val,cols,depth,lr)
        score = metrics.QLIKE(gb,val["target"])
            

        print(f"for lr={lr}, depth = {depth}, the score is {score}")



In [ ]:
import matplotlib.pyplot as plt
# 1. Final training set = train + val (1990–2022). Test = 2023 onward.
full = pd.concat([train, val])
test_start = test.index[0]

HAR_COLS   = ["past_5_days", "past_22_days", "today_vol"]
FINAL_COLS = HAR_COLS + ["high_low_1", "vix"]

# 2. Fit every model ONCE and forecast the whole test period
preds = {
    "Naive (5d)":    baselines.naive(log_returns),                       
    "Rolling (22d)": baselines.rolling(log_returns),                      
    "EWMA":          baselines.ewm(log_returns),                          
    "GARCH":         baselines.garch(log_returns, test_start),     
    "HAR":           models.HAR(full, test,HAR_COLS),           
    "HAR + VIX":     models.HAR(full, test,FINAL_COLS),           
    "Random forest": models.random_forest(full, test, FINAL_COLS,leaf=65), 
    "XGBoost":       models.XGBOOST(full, test, FINAL_COLS, depth=1, lr=0.05),
    "XGBoost-QLIKE": models.XGBOOST_QLIKE(full, test, FINAL_COLS, depth=1, lr=0.01),
}
preds = {k: v.loc[test.index] for k, v in preds.items()}   

# 3. Overall + per-year scores, all from the same single fit
y = test["target"]
rows = []
for name, p in preds.items():
    row = {"model": name, "QLIKE": metrics.QLIKE(p, y), "RMSE": metrics.RMSE(p, y)} 
    for yr in sorted(set(test.index.year)):
        m = test.index.year == yr
        row[f"QLIKE {yr}"] = metrics.QLIKE(p[m], y[m])
    rows.append(row)
results = pd.DataFrame(rows).set_index("model").sort_values("QLIKE").round(4)
print(results)

# 4. April 2025 tariff shock: the test period's "2020"
win = slice("2025-03-01", "2025-05-31")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y.loc[win], color="black", label="Actual")
for name in ["GARCH", "HAR + VIX", "XGBoost-QLIKE"]:
    ax.plot(preds[name].loc[win], label=name)
ax.set_title("Forecasts vs actual, Mar–May 2025"); ax.legend(); plt.tight_layout(); plt.show()

print("Highest target in training:", full["target"].max().round(4))
print("Highest actual, 2025:      ", y.loc["2025"].max().round(4))
print("Highest XGB-QLIKE, 2025:   ", preds["XGBoost-QLIKE"].loc["2025"].max().round(4))

In [ ]:
fake = (full[FINAL_COLS].max() * 100).to_frame().T  
ceiling = models.XGBOOST_QLIKE(full, fake, FINAL_COLS, depth=1, lr=0.01)
print(ceiling)